# ML6402 – Practical 04: Regression I — Simple Linear Regression

**Instructor:** Dr. Ahmed Sabbah  
**Course:** Machine Learning (ML6402)

This practical is designed so that students can use the repository as a study resource, not only as a collection of code.

### Practical roadmap

**Part A — Understand the model**
1. Regression intuition
2. Scatter plot
3. Regression equation
4. Intercept and slope
5. Prediction
6. Residuals
7. Cost function
8. Cost versus slope
9. Gradient Descent
10. Learning rate

**Part B — Build the model with Scikit-Learn**
1. Load and inspect the real dataset
2. Create X and y
3. Split train/test
4. Fit `LinearRegression`
5. Inspect learned parameters
6. Predict
7. Visualize train/test results
8. Evaluate with MAE, MSE, RMSE, and R²
9. Interpret the model as a baseline

> **Key idea:** understand what the model is doing before relying on the library implementation.

## Part A — Understand Simple Linear Regression

Simple Linear Regression predicts one **continuous numerical target** using one input feature.

The model is:

[
hat{y} = phi_0 + phi_1 x
]

where:

- (x) = input feature
- (hat{y}) = predicted target
- (phi_0) = intercept
- (phi_1) = slope / coefficient

The goal is to learn values of (phi_0) and (phi_1) that produce useful predictions.

### A1. Start with a tiny dataset

We first use a deliberately simple dataset:

| x | y |
|---:|---:|
| 1 | 1 |
| 2 | 2 |
| 3 | 3 |

This example makes the mathematics visible before we move to the real stipend dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

x_toy = np.array([1.0, 2.0, 3.0])
y_toy = np.array([1.0, 2.0, 3.0])

toy = pd.DataFrame({"x": x_toy, "y": y_toy})
toy

### A2. Visualize the observations

A scatter plot helps us judge whether a roughly linear relationship is plausible.

In [ ]:
plt.scatter(x_toy, y_toy)
plt.title("Toy Dataset: x versus y")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

### A3. What does a candidate regression line look like?

For now, set the intercept to zero:

[
hat{y} = phi_1 x
]

We can compare several possible slopes.

In [ ]:
candidate_slopes = [0.0, 0.5, 1.0, 1.5]

plt.scatter(x_toy, y_toy, label="Observed data")

x_line = np.linspace(0, 3.2, 100)

for slope in candidate_slopes:
    plt.plot(x_line, slope * x_line, label=f"slope = {slope}")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Different Candidate Regression Lines")
plt.legend()
plt.show()

### A4. Prediction

Suppose:

[
phi_0 = 30{,}000,qquad phi_1 = 8{,}500
]

Then for 5 years of experience:

[
hat{y} = 30{,}000 + 8{,}500(5)
]

The model predicts 72,500.

The important point is that **prediction is easy after the parameters have been learned**.

In [ ]:
phi0 = 30000
phi1 = 8500
experience = 5

prediction = phi0 + phi1 * experience
print("Predicted stipend:", prediction)

### A5. Residual — error for one observation

A residual measures the difference between the actual and predicted value:

[
residual_i = y_i - hat{y}_i
]

- positive residual → prediction is below the actual value;
- negative residual → prediction is above the actual value.

In [ ]:
actual = 60000
predicted = 55000

residual = actual - predicted

print("Actual:", actual)
print("Predicted:", predicted)
print("Residual:", residual)

### A6. Why do we not simply add residuals?

Positive and negative residuals can cancel.

For example:

- one residual = +5
- another residual = −5
- sum = 0

The predictions are not perfect, even though the sum is zero.

A squared-error cost prevents this cancellation.

In [ ]:
residuals = np.array([5, -5])

print("Residual sum:", residuals.sum())
print("Squared errors:", residuals**2)
print("Sum of squared errors:", np.sum(residuals**2))

## A7. Cost function

The lecture uses:

[
J(phi_0,phi_1)
=
rac{1}{2n}
sum_{i=1}^{n}
left(h_phi(x_i)-y_iight)^2
]

The extra (1/2) is useful in the derivative and does not change the location of the minimum.

For the first demonstration, we keep (phi_0=0) and vary only (phi_1).

In [ ]:
def lecture_cost(x, y, phi0, phi1):
    predictions = phi0 + phi1 * x
    errors = predictions - y
    return np.sum(errors ** 2) / (2 * len(x))

for slope in [0.0, 0.5, 1.0]:
    cost = lecture_cost(x_toy, y_toy, phi0=0, phi1=slope)
    print(f"slope={slope:>3}  cost={cost:.3f}")

The expected pattern is:

- (phi_1=0) → high cost
- (phi_1=0.5) → lower cost
- (phi_1=1) → zero cost for this toy dataset

So the cost gives an objective definition of a better line.

### A8. Plot slope against cost

Now calculate the cost for many candidate slopes and visualize the objective.

In [ ]:
slopes = np.linspace(-0.5, 2.5, 200)
costs = [lecture_cost(x_toy, y_toy, 0, s) for s in slopes]

plt.plot(slopes, costs)
plt.xlabel("Candidate slope φ1")
plt.ylabel("Cost J")
plt.title("Cost as a Function of the Slope")
plt.show()

The lowest point is the slope that minimizes the training objective.

For this toy example, the minimum occurs at approximately (phi_1=1).

## A9. Gradient Descent

Gradient Descent repeatedly updates the parameters:

[
phi_j leftarrow phi_j - alpha rac{partial J}{partial phi_j}
]

where (alpha) is the **learning rate**.

For Simple Linear Regression:

[
rac{partial J}{partial phi_0}
=
rac{1}{n}sum(h_phi(x_i)-y_i)
]

[
rac{partial J}{partial phi_1}
=
rac{1}{n}sum(h_phi(x_i)-y_i)x_i
]

Both parameter updates should use gradients computed from the same old parameter values.

### A10. One Gradient Descent iteration manually

Use:

- data: (1,1), (2,2), (3,3)
- start: (phi_0=0), (phi_1=0)
- learning rate: (alpha=0.1)

In [ ]:
phi0 = 0.0
phi1 = 0.0
alpha = 0.1
n = len(x_toy)

pred = phi0 + phi1 * x_toy
errors = pred - y_toy

grad_phi0 = np.sum(errors) / n
grad_phi1 = np.sum(errors * x_toy) / n

new_phi0 = phi0 - alpha * grad_phi0
new_phi1 = phi1 - alpha * grad_phi1

print("Predictions:", pred)
print("Errors:", errors)
print("Gradient phi0:", grad_phi0)
print("Gradient phi1:", grad_phi1)
print("Updated phi0:", new_phi0)
print("Updated phi1:", new_phi1)
print("Old cost:", lecture_cost(x_toy, y_toy, phi0, phi1))
print("New cost:", lecture_cost(x_toy, y_toy, new_phi0, new_phi1))

The cost should decrease after the update.

This is the basic idea of Gradient Descent: **use the gradient to correct the parameters iteratively**.

### A11. Implement Gradient Descent from scratch

The following function records the cost at every iteration so that we can study convergence.

In [ ]:
def gradient_descent(x, y, alpha=0.1, iterations=50):
    phi0 = 0.0
    phi1 = 0.0
    history = []

    n = len(x)

    for iteration in range(iterations):
        predictions = phi0 + phi1 * x
        errors = predictions - y

        cost = np.sum(errors ** 2) / (2 * n)
        history.append((iteration, phi0, phi1, cost))

        grad_phi0 = np.sum(errors) / n
        grad_phi1 = np.sum(errors * x) / n

        temp0 = phi0 - alpha * grad_phi0
        temp1 = phi1 - alpha * grad_phi1

        phi0 = temp0
        phi1 = temp1

    return phi0, phi1, pd.DataFrame(
        history,
        columns=["Iteration", "phi0", "phi1", "Cost"]
    )

phi0_gd, phi1_gd, history = gradient_descent(
    x_toy,
    y_toy,
    alpha=0.1,
    iterations=50
)

print("Final phi0:", phi0_gd)
print("Final phi1:", phi1_gd)
history.head()

In [ ]:
plt.plot(history["Iteration"], history["Cost"])
plt.xlabel("Iteration")
plt.ylabel("Cost J")
plt.title("Gradient Descent Convergence")
plt.show()

### A12. Learning rate experiment

The learning rate controls step size.

- too small → slow convergence;
- appropriate → stable progress;
- too large → may overshoot or diverge.

We can see this directly by trying different values.

In [ ]:
learning_rates = [0.01, 0.1, 0.5]

for lr in learning_rates:
    _, _, h = gradient_descent(x_toy, y_toy, alpha=lr, iterations=25)
    plt.plot(h["Iteration"], h["Cost"], label=f"alpha={lr}")

plt.xlabel("Iteration")
plt.ylabel("Cost J")
plt.title("Effect of Learning Rate")
plt.legend()
plt.show()

### Part A checkpoint

Before moving to the library implementation, make sure you can explain:

1. What do (x), (y), and (hat{y}) represent?
2. What does the intercept mean?
3. What does the slope mean?
4. What is a residual?
5. Why do we square errors?
6. What does the cost function measure?
7. What does Gradient Descent update?
8. What does the learning rate control?

---

# Part B — Simple Linear Regression with the Stipend Dataset

The course example contains **30 observations** relating research experience to stipend.

The source dataset uses the column name `ReserachExperience` (a spelling used in the original CSV). In this notebook we rename it to `ResearchExperience` after loading so that the code is clearer.

## B1. Load the dataset

Choose **one method only**.

- Local Anaconda/Jupyter: read from the repository's `data` folder.
- Google Colab: read the same CSV directly from GitHub.

In [ ]:
from pathlib import Path

# ============================================================
# DATASET LOADING – CHOOSE ONE METHOD ONLY
# ============================================================

# METHOD 1: Local Anaconda / Jupyter
DATA_PATH = Path("data") / "StipendData.csv"
dataset = pd.read_csv(DATA_PATH)

# METHOD 2: Google Colab
# Comment Method 1 above and uncomment the following two lines:
# DATA_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture04_Simple_Linear_Regression/data/StipendData.csv"
# dataset = pd.read_csv(DATA_URL)

# Rename the original CSV header for clarity
dataset = dataset.rename(columns={"ReserachExperience": "ResearchExperience"})

dataset.head()

## B2. Inspect the data before modeling

Reuse the inspection habits from Lecture 3.

In [ ]:
print("Shape:", dataset.shape)
print("\nData types:")
print(dataset.dtypes)

print("\nMissing values:")
print(dataset.isna().sum())

print("\nSummary statistics:")
display(dataset.describe())

## B3. Plot the relationship

Before fitting a regression model, inspect whether the relationship looks roughly linear.

In [ ]:
plt.scatter(
    dataset["ResearchExperience"],
    dataset["Stipend"]
)

plt.xlabel("Years of Research Experience")
plt.ylabel("Stipend")
plt.title("Research Experience versus Stipend")
plt.show()

### Interpretation question

Does the scatter plot suggest a positive or negative relationship?

Does the relationship look roughly linear enough for Simple Linear Regression to be a reasonable first baseline?

## B4. Create X and y

- `X` is the feature matrix.
- `y` is the target vector.

Double brackets keep `X` two-dimensional, as expected by Scikit-Learn.

In [ ]:
X = dataset[["ResearchExperience"]]
y = dataset["Stipend"]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

## B5. Train/test split

The lecture follows the book example:

- training: 2/3
- testing: 1/3
- `random_state=0`

The test data must remain unseen while the model is fitted.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=1/3,
    random_state=0
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## B6. Fit `LinearRegression`

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model fitted.")

## B7. Inspect the learned intercept and slope

The fitted equation is:

[
hat{y} = phi_0 + phi_1 x
]

Scikit-Learn stores:

- `intercept_` → (phi_0)
- `coef_[0]` → (phi_1)

In [ ]:
intercept = model.intercept_
slope = model.coef_[0]

print("Intercept:", intercept)
print("Slope:", slope)
print(f"Fitted equation: Stipend = {intercept:.2f} + {slope:.2f} × ResearchExperience")

### Interpretation

Write the slope in domain language:

> For each additional year of research experience, the model predicts that stipend changes by approximately **[slope] units**.

Also ask whether the intercept at zero years has a meaningful practical interpretation.

## B8. Make a single prediction

Predict the stipend for a researcher with 5 years of experience.

In [ ]:
new_experience = pd.DataFrame({"ResearchExperience": [5.0]})
predicted_stipend = model.predict(new_experience)[0]

print("Predicted stipend for 5 years:", predicted_stipend)

## B9. Predict the test set

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "ResearchExperience": X_test["ResearchExperience"].values,
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison["Residual"] = comparison["Actual"] - comparison["Predicted"]
comparison

## B10. Visualize the fitted line on training data

The points are observed training examples.

The line shows the model's predictions.

In [ ]:
plt.scatter(X_train["ResearchExperience"], y_train)

x_line = np.linspace(
    X_train["ResearchExperience"].min(),
    X_train["ResearchExperience"].max(),
    100
)

line_df = pd.DataFrame({"ResearchExperience": x_line})
y_line = model.predict(line_df)

plt.plot(x_line, y_line)

plt.title("Training Data and Fitted Regression Line")
plt.xlabel("Years of Research Experience")
plt.ylabel("Stipend")
plt.show()

## B11. Visualize the learned line on test data

The regression line is still the line learned from the **training set**.

The test points were not used to estimate the line.

In [ ]:
plt.scatter(X_test["ResearchExperience"], y_test)
plt.plot(x_line, y_line)

plt.title("Test Data and Learned Regression Line")
plt.xlabel("Years of Research Experience")
plt.ylabel("Stipend")
plt.show()

## B12. Residual plot

Residuals help us inspect prediction errors.

A residual plot complements the numerical metrics.

In [ ]:
residuals = y_test.values - y_pred

plt.scatter(y_pred, residuals)
plt.axhline(0)
plt.xlabel("Predicted Stipend")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.show()

## B13. Regression evaluation metrics

We calculate:

### MAE
Average absolute prediction error.

### MSE
Average squared prediction error.

### RMSE
Square root of MSE; returned to the same unit as the target.

### R²
Performance relative to predicting the target mean.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

### Metric interpretation

- Lower MAE, MSE, and RMSE indicate smaller prediction errors.
- MAE and RMSE are in the same target unit as stipend.
- MSE is in squared units.
- Higher R² generally indicates stronger performance relative to the mean baseline.
- R² does **not** prove causality.

## B14. Compare against a mean baseline

R² is related to the idea of comparing the regression model to a simple baseline that always predicts the mean target value.

We calculate that baseline explicitly.

In [ ]:
mean_prediction = np.full(
    shape=len(y_test),
    fill_value=y_train.mean()
)

baseline_rmse = np.sqrt(mean_squared_error(y_test, mean_prediction))
model_rmse = rmse

print("Mean-baseline RMSE:", baseline_rmse)
print("Linear-model RMSE:", model_rmse)

### Research interpretation

A simple linear model is useful as a **baseline** because it is:

- easy to reproduce;
- easy to interpret;
- easy to compare against more complex methods.

A more complex model should justify its extra complexity through better predictive performance or another meaningful advantage.

## Common mistakes

Avoid these mistakes:

1. Fit the model before creating the train/test split.
2. Interpret slope without mentioning units.
3. Ignore the scatter plot.
4. Evaluate only on training data.
5. Report only R² without an error metric.
6. Treat a predictive relationship as proof of causation.
7. Hide the random seed or data version.

## Student exercises

### Exercise 1 — Model interpretation
Using the fitted model:
1. report the intercept;
2. report the slope;
3. explain both in domain language;
4. predict stipend at 6.5 years.

### Exercise 2 — Residuals
1. identify the test observation with the largest absolute residual;
2. report its actual and predicted values;
3. explain why residual sign alone is not enough to measure error magnitude.

### Exercise 3 — Metrics
Calculate MAE, MSE, RMSE, and R² manually using NumPy formulas, then compare them with Scikit-Learn's results.

### Exercise 4 — Learning rate
Run the toy Gradient Descent example with:
- (alpha=0.01)
- (alpha=0.1)
- a much larger value

Describe the convergence behavior.

### Exercise 5 — Reproducibility
Change `random_state` in the train/test split. Compare the test metrics and explain why the result changes.

## Challenge — Build a regression baseline on another dataset

Choose another dataset with:

- one numerical predictor;
- one continuous numerical target.

Your notebook must include:

1. `head()`, `info()`, and `describe()`;
2. a scatter plot;
3. train/test split;
4. fitted intercept and slope;
5. predicted values;
6. residuals;
7. training and test plots;
8. MAE;
9. RMSE;
10. R²;
11. two sentences interpreting the model in domain language;
12. a statement explaining whether the model is a useful baseline.

## Final self-check

You should now be able to explain:

1. What is regression?
2. What is the difference between (X), (y), and (hat{y})?
3. What do (phi_0) and (phi_1) represent?
4. What is a residual?
5. Why do we need a cost function?
6. Why do squared errors not cancel?
7. What does Gradient Descent update?
8. What does the learning rate control?
9. Why do we keep test data unseen?
10. How do MAE, MSE, RMSE, and R² differ?
11. Why is Simple Linear Regression useful as a research baseline?

---

### Source note

The stipend dataset and the core implementation sequence are aligned with the course textbook example used in Lecture 4. The notebook extends the example with additional visual explanations, a from-scratch Gradient Descent demonstration, residual analysis, explicit baseline comparison, and study exercises.

**ML6402 – Machine Learning**  
**Instructor: Dr. Ahmed Sabbah**